# **Method (02) tf.data API**
- Modern
- Most powerful + flexible pipeline system
- Supports:
  - large datasets
  - streaming
  - performance optimization (prefetch, cache, map)

In [37]:
import tensorflow as tf
import numpy as np

In [38]:
# laod data as file paths(strings) into a tf dataset
images = tf.data.Dataset.list_files("data/*/*", shuffle=False)

In [39]:
len(images)

2152

In [40]:
for i in images.take(3):
    test_path = i
    print(i)

tf.Tensor(b'data\\Potato___Early_blight\\001187a0-57ab-4329-baff-e7246a9edeb0___RS_Early.B 8178.JPG', shape=(), dtype=string)
tf.Tensor(b'data\\Potato___Early_blight\\002a55fb-7a3d-4a3a-aca8-ce2d5ebc6925___RS_Early.B 8170.JPG', shape=(), dtype=string)
tf.Tensor(b'data\\Potato___Early_blight\\009c8c31-f22d-4ffd-8f16-189c6f06c577___RS_Early.B 7885.JPG', shape=(), dtype=string)


In [41]:
import os

classes = os.listdir("data")
classes

['Potato___Early_blight', 'Potato___healthy', 'Potato___Late_blight']

In [42]:
def get_label(path):
    return tf.strings.split(path, os.path.sep)[-2]

In [43]:
test_label = get_label(test_path)
test_label

<tf.Tensor: shape=(), dtype=string, numpy=b'Potato___Early_blight'>

## Data splitting

In [44]:
image_count = len(images)
print(image_count)

train_size = int(image_count*0.7)
print(train_size)

val_size = int(image_count*0.2)
print(val_size)

test_size = int(image_count*0.1)
print(test_size)

2152
1506
430
215


In [45]:
train_images = images.take(train_size)
val_images = images.skip(train_size).take(val_size)
test_images = images.skip(train_size).skip(val_size)

print(len(train_ds), len(val_ds), len(test_ds))

1506 430 216


In [46]:
def process_train_image(path):
    label = get_label(path)

    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [128, 128]) # does not change the channel dimension at all — it only changes height and width
    img = img/255 # scaling 0-1

    # augmentation
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.1)
    img = tf.image.random_contrast(img, 0.9, 1.1)

    return img, label

In [47]:
def process_val_test_image(path):
    label = get_label(path)

    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [128, 128]) # does not change the channel dimension at all — it only changes height and width
    img = img/255 # scaling 0-1
    
    return img, label

In [48]:
# all ds are now TF Dataset and each item is a tensors

train_ds = train_images.map(process_train_image) 
test_ds = test_images.map(process_val_test_image)
val_ds = val_images.map(process_val_test_image)

In [51]:
len(train_ds)

1506

In [52]:
for i in train_ds.take(1):
    print(i[0].shape, i[1].numpy()) # since large size of mtrices check shape 

(128, 128, 3) b'Potato___Early_blight'


## Shuffling (training dataset only)

In [53]:
train_ds = train_ds.shuffle(buffer_size=3000) # This is not a one time process this is an instruction to Dataset, so it shuffle in each epoch

In [54]:
for i, l in train_ds.take(10):
    # to see clearly convert into numpy obj
    print(i.shape, l.numpy())
    print(i[0][0].numpy(), "\n") # first row, frist column values(3 channels)

(128, 128, 3) b'Potato___Early_blight'
[0.7846581 0.7619677 0.7819631] 

(128, 128, 3) b'Potato___Early_blight'
[0.49463716 0.4687556  0.4922294 ] 

(128, 128, 3) b'Potato___Early_blight'
[0.7826934  0.77314174 0.7886572 ] 

(128, 128, 3) b'Potato___Early_blight'
[0.5093039  0.5172056  0.57009387] 

(128, 128, 3) b'Potato___Early_blight'
[0.5683366  0.55529356 0.59329444] 

(128, 128, 3) b'Potato___Early_blight'
[0.59369653 0.58629775 0.65272677] 

(128, 128, 3) b'Potato___Early_blight'
[0.61466736 0.60674214 0.6581317 ] 

(128, 128, 3) b'Potato___Late_blight'
[0.64748895 0.619958   0.6421014 ] 

(128, 128, 3) b'Potato___Early_blight'
[0.65419924 0.64721    0.6860659 ] 

(128, 128, 3) b'Potato___Late_blight'
[0.58155686 0.5287077  0.5261564 ] 



## Batching

In [55]:
train_ds = train_ds.batch(32)
test_ds = test_ds.batch(32)
val_ds = val_ds.batch(32)

In [56]:
type(train_ds)

tensorflow.python.data.ops.batch_op._BatchDataset

In [57]:
len(train_ds)

48

In [58]:
# first batch of Dataset
# iterate thorugh tuples each tuple has two batches as (img_batch, label_batch) in size 32

for i in train_ds.take(1):
    print(i[0].shape, "\n", i[1].numpy())

(32, 128, 128, 3) 
 [b'Potato___Early_blight' b'Potato___Early_blight'
 b'Potato___Early_blight' b'Potato___Early_blight'
 b'Potato___Early_blight' b'Potato___Late_blight' b'Potato___Late_blight'
 b'Potato___Early_blight' b'Potato___Early_blight'
 b'Potato___Early_blight' b'Potato___Early_blight'
 b'Potato___Early_blight' b'Potato___Early_blight' b'Potato___Late_blight'
 b'Potato___Late_blight' b'Potato___Early_blight' b'Potato___Early_blight'
 b'Potato___Late_blight' b'Potato___Early_blight' b'Potato___Early_blight'
 b'Potato___Late_blight' b'Potato___Early_blight' b'Potato___Early_blight'
 b'Potato___Early_blight' b'Potato___Late_blight' b'Potato___Early_blight'
 b'Potato___Early_blight' b'Potato___Early_blight'
 b'Potato___Early_blight' b'Potato___Early_blight' b'Potato___Late_blight'
 b'Potato___Late_blight']


## Performance optimization
- AUTOTUNE is TensorFlow’s way of automatically optimizing CPU/GPU pipeline performance so don’t need to manually tune threads or prefetch sizes
- Decides how many CPU threads to use
- Controls parallel execution in .map()
- Decides how many batches to load ahead of time
- Keep GPU/CPU always busy by preparing next batch early
- Ensures GPU is not waiting for data loading
- Remove manual tuning effort no need to set:
  - number of threads
  - prefetch buffer size
  - parallel workers

In [59]:
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

### since we added scaling, augmentations in image preprocessing dont add scaling layer or augmentation layer in CNN architecture